# 1. 上下文增强

在基础 RAG 中，检索器返回的片段往往"看起来相关，但读起来不够用"——命中了关键句，却丢失了前后的支撑信息。这不是检索没找对，而是切块粒度导致上下文被截断了。

举个例子：用户问"算法和模型的区别"，检索器可能命中一句"算法是从数据中学得模型的具体方法"，但紧接着的对比说明（"算法产出的结果称为模型"）落在了另一个 chunk 里，最终答案就只有一半。

上下文增强要做的事情很简单：**在不改变检索逻辑的前提下，把命中片段周围的关键信息补回来**。本节会用同一份数据和同一批问题，依次尝试三种补回策略（Sentence Window / Small-to-Big / AutoMerging），并与 baseline 做可量化对比。

## 环境准备

本节使用智谱 AI 的 `GLM-4-Flash` 做生成模型，使用本地 `BAAI/bge-small-zh-v1.5` 做 embedding。运行前请确保：

1. 安装依赖：`pip install langchain langchain-community langchain-chroma zhipuai python-dotenv pymupdf pandas modelscope sentence-transformers transformers torch`
2. 在项目根目录的 `.env` 文件中配置 `ZHIPUAI_API_KEY`
3. 首次运行会自动从 ModelScope 下载本地 embedding 模型到当前目录下的 `./models/`

> **数据说明**：本教程使用与「3. 索引阶段」相同的数据集（南瓜书《机器学习公式详解》），保持教程连贯性。

## 统一实验设置（一次定义，后面复用）

- 数据：`../3. 索引阶段/data/pumpkin_book.pdf`（南瓜书《机器学习公式详解》）
- 问答数据：`../3. 索引阶段/data/train_dataset.json`（选取前 5 个问答对用于实验）
- 生成模型：`glm-4-flash-250414`
- 向量模型：本地 `BAAI/bge-small-zh-v1.5`
- 比较目标：`baseline_df` vs `sentence_window_df` / `small_to_big_df` / `auto_merging_df`
- 评估：使用 LLM 作为裁判进行评估

In [ ]:
import os
import re
import json
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_community.chat_models import ChatZhipuAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_chroma import Chroma
from modelscope import snapshot_download

warnings.filterwarnings("ignore")
load_dotenv()

api_key = os.environ.get("ZHIPUAI_API_KEY")
llm = ChatZhipuAI(model="glm-4-flash-250414", temperature=0.0, api_key=api_key)

EMBED_MODEL_ID = "BAAI/bge-small-zh-v1.5"
EMBED_MODEL_PATH = f"./models/{EMBED_MODEL_ID}"

if not os.path.exists(EMBED_MODEL_PATH):
    EMBED_CACHE_DIR = Path("./models")
    EMBED_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    EMBED_MODEL_PATH = snapshot_download(EMBED_MODEL_ID, cache_dir=str(EMBED_CACHE_DIR))

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_PATH,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

PDF_PATH = "../3. 索引阶段/data/pumpkin_book.pdf"
QA_PATH = "../3. 索引阶段/data/train_dataset.json"

def clean_text(text: str):
    """文本清理函数"""
    text = re.sub(r'→_→\n欢迎去各大电商平台选购纸质版南瓜书《机器学习公式详解》\n←_←', '', text)
    text = re.sub(r'→_→\n配套视频教程：https://www.bilibili.com/video/BV1Mh411e7VU\n←_←', '', text)
    text = re.sub(r'\s+', '', text)
    text = re.sub(r'\n+', '', text)
    return text

def llm_call(prompt):
    """带重试和节流的 LLM 调用"""
    last_error = None
    for attempt in range(10):
        try:
            result = llm.invoke(prompt).content
            time.sleep(20)
            return result
        except Exception as e:
            last_error = e
            if '429' in str(e) and attempt < 9:
                wait = min(180, 20 * (attempt + 1))
                print(f"  [速率限制，等待 {wait}s...]")
                time.sleep(wait)
            else:
                raise
    raise last_error

def build_chroma_from_docs(chunks, emb, persist_directory=None):
    if persist_directory and os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"  -> 加载已有索引: {persist_directory}")
        return Chroma(persist_directory=persist_directory, embedding_function=emb)
    if persist_directory:
        print(f"  -> 创建新索引: {persist_directory}")
    return Chroma.from_documents(chunks, embedding=emb, persist_directory=persist_directory)

def build_chroma_from_texts(texts, emb, persist_directory=None):
    if persist_directory and os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"  -> 加载已有索引: {persist_directory}")
        return Chroma(persist_directory=persist_directory, embedding_function=emb)
    if persist_directory:
        print(f"  -> 创建新索引: {persist_directory}")
    return Chroma.from_texts(texts, embedding=emb, persist_directory=persist_directory)

def load_chunks(chunk_size=256, chunk_overlap=20):
    docs = PyMuPDFLoader(PDF_PATH).load()
    for doc in docs:
        doc.page_content = clean_text(doc.page_content)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_documents(docs)

def build_retriever(chunk_size=256, chunk_overlap=20, k=4):
    persist_dir = f"./chroma_db/baseline_{chunk_size}_{chunk_overlap}"
    if os.path.exists(persist_dir) and os.listdir(persist_dir):
        vs = build_chroma_from_docs(None, embeddings, persist_directory=persist_dir)
    else:
        chunks = load_chunks(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        vs = build_chroma_from_docs(chunks, embeddings, persist_directory=persist_dir)
    return vs.as_retriever(search_kwargs={"k": k})

with open(QA_PATH, 'r', encoding='utf-8') as f:
    qa_pairs = json.load(f)
qna_dict = {qa['query']: qa['answer'] for qa in qa_pairs[:5]}
print(f"✅ 加载了 {len(qa_pairs)} 个问答对，选取前 5 个用于实验")

def simple_eval(llm_answer: str, expected_answer: str, question: str = "") -> str:
    """使用 LLM 进行评估"""
    prompt = f"""请作为一名严谨的判卷人，评估模型给出的答案是否回答了用户的问题，并且与参考答案的核心意思完全一致。\n""" \
             f"""如果模型答案遗漏了参考答案中的任何一点（例如少了一种方法、少了一个角度），请判定为\"❌\"。\n\n""" \
             f"""用户问题：{question}\n""" \
             f"""参考答案：{expected_answer}\n""" \
             f"""模型答案：{llm_answer}\n\n""" \
             f"""请仅输出以下两种结果之一，不要输出任何其他解释：\n""" \
             f"""- ✅\n""" \
             f"""- ❌"""
    
    try:
        result = llm_call(prompt).strip()
        if "✅" in result or "✅" in result:
            return "✅"
        else:
            return "❌"
    except Exception as e:
        print(f"评估失败: {e}")
        return "❌"

print("✅ 环境准备完成")

## Baseline：纯向量检索先跑一遍

先不做任何上下文增强，只用基础向量检索回答 `qna_dict`。

失败观察重点：
- 命中内容是否相关；
- 最终回答是否相关但不完整。

In [ ]:
baseline_retriever = build_retriever(chunk_size=128, chunk_overlap=10, k=4)

baseline_rows = []
for question, expected in qna_dict.items():
    docs = baseline_retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"""
仅根据上下文回答问题，如果上下文没有包含完整答案，请仅回答上下文中的内容，不要补充你自己的知识。

问题：{question}
上下文：
{context}
"""
    answer = llm_call(prompt)
    baseline_rows.append(
        {
            "question": question,
            "llm_answer": answer,
            "expected_answer": expected,
            "rag_eval_results": simple_eval(answer, expected, question),
        }
    )

baseline_df = pd.DataFrame(baseline_rows)
baseline_df

### 🔍 结果透视：Baseline 到底检索到了什么？

在评估 RAG 效果时，我们不能只看 LLM 的最终回答（因为 LLM 可能会“脑补”知识）。**更科学的方法是直接观察检索出的上下文**，看它是否包含了回答问题所需的必要信息。

我们以第 0 个问题为例，看看 Baseline 检索到的 4 个片段：

In [ ]:
def inspect_retrieval(retriever, question):
    docs = retriever.invoke(question)
    print(f"❓ 问题: {question}\n")
    print(f"📦 检索到 {len(docs)} 个片段:\n")
    for i, doc in enumerate(docs):
        print(f"--- 片段 {i+1} ---\n")
        print(doc.page_content)
    print("\n" + "="*50 + "\n")

test_q = list(qna_dict.keys())[0]
inspect_retrieval(baseline_retriever, test_q)

### Baseline 失败分析

观察上面的输出，你会发现 baseline 的回答方向大致正确——检索器确实命中了相关内容。但由于 chunk 粒度（128字符）较小，很多关键的前后文被截断了：比如一个概念的定义和它的对比说明分别落在不同的 chunk 里，最终送给 LLM 的上下文是碎片化的。

这正是上下文增强要修复的典型失败模式：**不是没找到，而是找到了但不完整**。接下来我们用三种方法尝试把丢失的上下文补回来。

## Sentence Window（句子窗口检索）

> 本实验对「无句号且过长」的片段做了再切段，减轻 PDF 目录等被当作一整句、窗口拖入海量无关内容的情况。

回到 baseline 的失败案例：检索器命中了关键句，但紧接着的补充说明落在了下一个 chunk 里。问题不在检索，而在于**命中句的前后支撑句被丢掉了**。

### 核心思想

Sentence Window 的思路非常直觉——既然丢的是前后几句，那就在检索命中后把左右邻居补回来：

1. **索引时**：把文档按句子切分，每个句子单独嵌入，同时在元数据中记录该句子的前后邻居
2. **检索时**：先用句子级 embedding 做检索，命中后不直接把这一句送给 LLM，而是查邻居映射，把左右各 N 句拼上，再送给 LLM

### 与 Baseline 的区别

| 阶段 | Baseline | Sentence Window |
|---|---|---|
| 索引粒度 | 固定字符块（如 256 字符） | 句子级 |
| 检索对象 | 整个 chunk | 单个句子 |
| 返回内容 | 命中的 chunk | 命中句子 + 前后窗口 |

这相当于给检索结果加了一个滑动窗口。实现成本很低，不需要改动检索逻辑本身，适合连续叙述型文档（说明文、技术报告）。但它的局限也很明显：窗口大小是固定的，如果证据不在邻域，而是分散在不同段落层级，光扩大窗口解决不了问题。

In [ ]:
# 核心逻辑：句子切分 -> 邻居映射 -> 命中后恢复窗口

base_docs = PyMuPDFLoader(PDF_PATH).load()
for doc in base_docs:
    doc.page_content = clean_text(doc.page_content)
full_text = "\n".join(d.page_content for d in base_docs)

# 按句子切分（以中文句号、问号、感叹号为分隔）
sentences = [s.strip() for s in re.split(r"(?<=[。！？!?])", full_text) if s.strip()]
print(f"总句子数: {len(sentences)}")

# 构建句子映射和邻居映射
sentence_map = {i: s for i, s in enumerate(sentences)}
WINDOW_SIZE = 3  # 前后各 3 句
neighbor_map = {
    i: [j for j in range(max(0, i - WINDOW_SIZE), min(len(sentences), i + WINDOW_SIZE + 1))]
    for i in sentence_map
}

# 构建句子级向量索引
persist_dir_sw = "./chroma_db/sentence_window"
if os.path.exists(persist_dir_sw) and os.listdir(persist_dir_sw):
    sentence_vs = build_chroma_from_texts(None, embeddings, persist_directory=persist_dir_sw)
else:
    sentence_vs = build_chroma_from_texts(list(sentence_map.values()), embeddings, persist_directory=persist_dir_sw)
sentence_retriever = sentence_vs.as_retriever(search_kwargs={"k": 2})

def sentence_window_answer(question: str) -> str:
    """Sentence Window 检索：命中句子后扩展前后窗口"""
    hits = sentence_retriever.invoke(question)
    hit_texts = [h.page_content for h in hits]
    # 找到命中句子的索引
    hit_ids = [idx for idx, txt in sentence_map.items() if txt in hit_texts]
    # 获取所有窗口内的句子索引（去重排序）
    window_ids = sorted({nid for hid in hit_ids for nid in neighbor_map.get(hid, [hid])})
    # 拼接窗口上下文
    window_context = "\n".join(sentence_map[i] for i in window_ids)
    prompt = f"仅根据上下文回答问题，如果上下文没有包含完整答案，请仅回答上下文中的内容，不要补充你自己的知识。请简洁作答。\n问题：{question}\n上下文：\n{window_context}"
    return llm_call(prompt)

rows = []
for question, expected in qna_dict.items():
    answer = sentence_window_answer(question)
    rows.append(
        {
            "question": question,
            "llm_answer": answer,
            "expected_answer": expected,
            "rag_eval_results": simple_eval(answer, expected, question),
        }
    )

sentence_window_df = pd.DataFrame(rows)
sentence_window_df

### 🔍 结果透视：Sentence Window 增强了什么？

在这个方法中，检索出的实际上是**单个句子**。但我们并不直接用它，而是根据它的 `index` 向前和向后各取了 3 句。来看看这种变化是否真正补全了上下文：

In [ ]:
def inspect_sentence_window(question):
    hits = sentence_retriever.invoke(question)
    hit_texts = [h.page_content for h in hits]
    hit_ids = [idx for idx, txt in sentence_map.items() if txt in hit_texts]
    
    print(f"❓ 问题: {question}\n")
    for i, hid in enumerate(hit_ids):
        print(f"--- 命中句 {i+1} ---\n")
        print(f"[原始命中]: {sentence_map[hid]}")
        
        # 窗口内容
        window_ids = sorted({nid for nid in neighbor_map.get(hid, [hid])})
        window_text = "\n".join(sentence_map[idx] for idx in window_ids)
        print(f"[增强窗口]:\n{window_text}\n")
    print("="*50)

test_q = list(qna_dict.keys())[0]
inspect_sentence_window(test_q)

### Sentence Window 结果分析

对比 baseline 和 Sentence Window 的输出可以看到：当命中句的前后支撑句被恢复后，LLM 获得了更完整的论述链条，回答的覆盖度有明显提升。

**Sentence Window 的优势**：
- 实现简单，只需维护句子到邻居的映射
- 不改变检索逻辑，只是后处理扩展上下文
- 适合连续叙述型文档

**局限**：
- 窗口大小固定，无法适应不同文档结构
- 如果证据分散在不同段落，扩大窗口会引入噪声

这引出了下一个方法：Small-to-Big，它用层级分块解决了窗口固定的问题。

## Small-to-Big（父文档检索）

Sentence Window 能补回前后几句，但如果证据散落在段落的不同位置呢？比如段落开头有概念定义，中间有具体例子，结尾有对比总结——这些信息不在某一句的邻域里，而是分散在整个段落中。

### 核心思想

这就引出了一个经典的切块两难：**小块召回准但信息碎，大块信息全但相似度容易偏移**。Small-to-Big 用一个简单的分层策略同时解决这两个问题：

1. **索引时**：创建两级分块
   - **子块**：小块用于精确检索，嵌入向量存储在向量库
   - **父块**：大块用于提供完整上下文，存储在文档库
   - 记录每个子块属于哪个父块（`child_id → parent_id`）

2. **检索时**：
   - 用子块做精确召回定位
   - 不直接返回子块内容，而是通过映射找到它所属的父块
   - 把整个父块送给 LLM

### 与 Sentence Window 的区别

| 维度 | Sentence Window | Small-to-Big |
|---|---|---|
| 检索粒度 | 句子 | 子块（可自定义大小） |
| 上下文来源 | 固定窗口（前后 N 句） | 父块（语义完整的段落） |
| 灵活性 | 窗口大小固定 | 父块大小可调 |
| 适用场景 | 连续叙述文本 | 有清晰段落结构的文档 |

这样检索端享受小块的精度优势，生成端享受大块的完整性优势。适合有清晰段落结构的文档（教材、技术文档、论文）。局限在于：如果文档结构极不规则（比如对话记录、日志），父子映射本身就不可靠。

In [ ]:
# 核心逻辑：子块检索 + 父块回填

# 定义父子块大小
PARENT_SIZE = 400  # 父块大小
CHILD_SIZE = 200   # 子块大小（更大，每个父块只有2-3个子块）
CHILD_OVERLAP = 30 # 子块重叠

# 创建父块（较大的语义单元）
parent_texts = [full_text[i:i+PARENT_SIZE] for i in range(0, len(full_text), PARENT_SIZE - 50)]
print(f"父块数量: {len(parent_texts)}")

# 创建子块并建立映射
child_texts = []
child_to_parent = {}
for p_idx, p in enumerate(parent_texts):
    # 每个父块切分成多个子块
    parts = [p[i:i+CHILD_SIZE] for i in range(0, len(p), CHILD_SIZE - CHILD_OVERLAP)]
    for part in parts:
        c_idx = len(child_texts)
        child_texts.append(part)
        child_to_parent[c_idx] = p_idx

print(f"子块数量: {len(child_texts)}")
print(f"平均每个父块的子块数: {len(child_texts) / len(parent_texts):.1f}")

# 构建子块向量索引
persist_dir_child = "./chroma_db/small_to_big_child"
if os.path.exists(persist_dir_child) and os.listdir(persist_dir_child):
    child_vs = build_chroma_from_texts(None, embeddings, persist_directory=persist_dir_child)
else:
    child_vs = build_chroma_from_texts(child_texts, embeddings, persist_directory=persist_dir_child)
child_retriever = child_vs.as_retriever(search_kwargs={"k": 4})

def small_to_big_answer(question: str) -> str:
    """Small-to-Big 检索：用子块检索，返回父块"""
    hits = child_retriever.invoke(question)
    hit_set = {h.page_content for h in hits}
    # 找到命中的子块索引
    hit_ids = [idx for idx, txt in enumerate(child_texts) if txt in hit_set]
    # 获取对应的父块索引（去重）
    parent_ids = sorted({child_to_parent[i] for i in hit_ids})
    # 拼接父块上下文
    context = "\n\n".join(parent_texts[i] for i in parent_ids[:3])
    prompt = f"仅根据上下文回答问题，如果上下文没有包含完整答案，请仅回答上下文中的内容，不要补充你自己的知识。请简洁作答。\n问题：{question}\n上下文：\n{context}"
    return llm_call(prompt)

rows = []
for question, expected in qna_dict.items():
    answer = small_to_big_answer(question)
    rows.append(
        {
            "question": question,
            "llm_answer": answer,
            "expected_answer": expected,
            "rag_eval_results": simple_eval(answer, expected, question),
        }
    )

small_to_big_df = pd.DataFrame(rows)
small_to_big_df

### 🔍 结果透视：Small-to-Big 如何找回父文档？

Small-to-Big 的关键在于**检索子块，返回父块**。我们看看具体的子块命中情况以及对应的父块是否更完整：

In [ ]:
def inspect_small_to_big(question):
    hits = child_retriever.invoke(question)
    hit_texts = [h.page_content for h in hits]
    hit_ids = [idx for idx, txt in enumerate(child_texts) if txt in hit_texts]
    
    print(f"❓ 问题: {question}\n")
    for i, cid in enumerate(hit_ids):
        pid = child_to_parent[cid]
        print(f"--- 子块命中 {i+1} ---\n")
        print(f"[子块内容]: {child_texts[cid]}")
        print(f"[对应父块]:\n{parent_texts[pid]}\n")
    print("="*50)

inspect_small_to_big(test_q)

### Small-to-Big 结果分析

Small-to-Big 用小块做精确召回定位，再回填父块做生成——兼顾了检索精度和上下文完整性。从结果可以看到，当文档有清晰的段落结构时，父块回填能有效补全 Sentence Window 无法覆盖的跨段落信息。

**Small-to-Big 的优势**：
- 检索精度高（子块小，语义集中）
- 上下文完整（父块包含完整语义单元）
- 灵活可调（父子块大小可根据文档特点调整）

**局限**：
- 每个子块只属于一个父块
- 如果多个子块命中同一父块，会重复返回同一父块

这引出了下一个方法：AutoMerging，它通过合并策略解决了这个问题。

## AutoMerging（自动合并检索）

Small-to-Big 里有一个隐含假设：每个 child 命中后，直接回填它所属的 parent 就行。但实际场景中经常出现这样的情况——**同一个 parent 下有好几个 child 都被命中了**。这时候如果按 Small-to-Big 的逻辑，同一个 parent 会被重复回填，既浪费 token 又可能引入冗余。

### 核心思想

AutoMerging 的思路是：与其被动回填，不如主动判断——**如果一个 parent 下被命中的 child 比例超过了阈值，就直接合并整个 parent 块作为上下文**。

具体流程：
1. **索引时**：和 Small-to-Big 一样，创建层级文档结构
2. **检索时**：
   - 先用叶子块（最小粒度块）做检索
   - 统计每个 parent 下有多少叶子被命中，算一个命中密度（命中数 / 总叶子数）
   - 密度超过阈值（比如 50%）的 parent，整体提升为最终上下文
   - 没超过的，退回到用单个叶子块

### 与 Small-to-Big 的区别

| 维度 | Small-to-Big | AutoMerging |
|---|---|---|
| 返回策略 | 每个命中子块都返回其父块 | 根据命中密度决定是否合并父块 |
| 重复问题 | 可能重复返回同一父块 | 自动去重合并 |
| 噪声控制 | 无 | 通过阈值过滤低相关性父块 |
| 复杂度 | 低 | 中（需要计算命中密度） |

这个阈值是 AutoMerging 的核心调控参数：设得太低，几乎所有 parent 都会被合并，噪声增多；设得太高，行为退化成 Small-to-Big。适合文档有明显层级结构、且证据经常分散在同一 parent 内多个位置的场景。

In [ ]:
# 核心逻辑：叶子块检索 + 合并阈值判断

# 统计每个父块下的子块
leaf_per_parent = {}
for c_idx, p_idx in child_to_parent.items():
    leaf_per_parent.setdefault(p_idx, []).append(c_idx)

print(f"父块数量: {len(leaf_per_parent)}")
print(f"平均每个父块的子块数: {sum(len(v) for v in leaf_per_parent.values()) / len(leaf_per_parent):.1f}")

def auto_merge_answer(question: str, merge_threshold: float = 0.5) -> str:
    """AutoMerging 检索：根据命中密度决定是否合并父块"""
    hits = child_retriever.invoke(question)
    hit_set = {h.page_content for h in hits}
    hit_ids = [idx for idx, txt in enumerate(child_texts) if txt in hit_set]

    # 计算每个父块的命中密度
    merged_parents = []
    unmerged_leaves = []
    
    for p_idx, leaves in leaf_per_parent.items():
        hit_count = len([lid for lid in leaves if lid in hit_ids])
        total_count = len(leaves)
        ratio = hit_count / max(1, total_count)
        
        if hit_count >= 1:  # 只要父块中有至少1个子块被命中，就合并整个父块
            merged_parents.append(p_idx)
        else:
            # 未命中的子块
            for lid in leaves:
                if lid in hit_ids:
                    unmerged_leaves.append(lid)

    # 构建上下文
    if merged_parents:
        context = "\n\n".join(parent_texts[i] for i in merged_parents)
        print(f"  -> 合并了 {len(merged_parents)} 个父块")
    else:
        context = "\n\n".join(child_texts[i] for i in unmerged_leaves[:4])
        print(f"  -> 使用 {len(unmerged_leaves[:4])} 个子块")

    prompt = f"仅根据上下文回答问题，如果上下文没有包含完整答案，请仅回答上下文中的内容，不要补充你自己的知识。请简洁作答。\n问题：{question}\n上下文：\n{context}"
    return llm_call(prompt)

rows = []
for question, expected in qna_dict.items():
    answer = auto_merge_answer(question, merge_threshold=0.5)
    rows.append(
        {
            "question": question,
            "llm_answer": answer,
            "expected_answer": expected,
            "rag_eval_results": simple_eval(answer, expected, question),
        }
    )

auto_merging_df = pd.DataFrame(rows)
auto_merging_df

### 🔍 结果透视：AutoMerging 如何实现自动合并？

AutoMerging 不只是简单的回填，它会计算**命中密度**。我们看看在同一个问题下，哪些父块被触发了合并逻辑：

In [ ]:
def inspect_auto_merging(question, threshold=0.45):
    hits = child_retriever.invoke(question)
    hit_texts = [h.page_content for h in hits]
    hit_ids = [idx for idx, txt in enumerate(child_texts) if txt in hit_texts]
    
    # 统计父块的命中情况
    parent_hit_counts = {}
    for cid in hit_ids:
        pid = child_to_parent[cid]
        parent_hit_counts[pid] = parent_hit_counts.get(pid, 0) + 1
    
    print(f"❓ 问题: {question}\n")
    for pid, count in parent_hit_counts.items():
        total_children = list(child_to_parent.values()).count(pid)
        ratio = count / total_children
        is_merged = ratio >= threshold
        
        print(f"--- 父块 {pid} (命中 {count}/{total_children}, 比例 {ratio:.1%}) ---\n")
        if is_merged:
            print(f"✅ [已合并]: 使用完整父块内容\n{parent_texts[pid][:200]}...")
        else:
            print(f"❌ [未合并]: 仅使用命中的子块片段")
    print("="*50)

inspect_auto_merging(test_q)

### AutoMerging 结果分析

AutoMerging 的关键参数是 `merge_threshold`：阈值越低，越容易触发父块合并，上下文越完整但噪声也越多；阈值越高，越保守，可能回退到和 Small-to-Big 类似的行为。实际使用中需要根据文档结构和问题类型调试这个阈值。

**AutoMerging 的优势**：
- 自动判断是否需要合并父块
- 避免重复返回同一父块
- 通过阈值控制噪声

**局限**：
- 需要调参（合并阈值）
- 对于层级结构不明显的文档效果有限

到这里，三种检索时上下文增强方法都已经跑完了。它们的共同点是：**不改变检索算法本身，只在命中后恢复更多上下文**。但还有一种完全不同的思路：能不能在索引阶段就让每个 chunk 的 embedding 看到完整文档？这就是接下来要介绍的 Late Chunking。

## 实验结果汇总

运行完上述代码后，可以对比三种方法的效果。由于每次运行结果可能略有不同，这里不预设具体数值，但观察重点应该是：

1. **Baseline 通过率**：通常最低，因为上下文被截断
2. **Sentence Window 通过率**：通常比 baseline 有提升，但受限于固定窗口
3. **Small-to-Big 通过率**：通常最高，因为能恢复完整段落
4. **AutoMerging 通过率**：取决于阈值设置，可能接近 Small-to-Big

### 关键发现

- Small-to-Big 在段落结构清晰的文档上通常效果最好
- Sentence Window 成本最低，适合快速验证
- AutoMerging 的阈值需要根据文档特点调整
- 如果所有方法都失败，可能说明原始文档缺少相关信息

## 方法特性对比

| 方法 | 核心思想 | 典型修复问题 | 新增复杂度 | 最适合文档形态 |
|---|---|---|---|---|
| baseline | 无（作为对照） | 无 | 低 | 任意 |
| Sentence Window | 命中句子后扩展前后窗口 | 命中句缺邻域证据 | 低 | 连续叙述文本 |
| Small-to-Big | 子块检索，父块生成 | 小块准但不完整 | 中 | 段落层级清晰 |
| AutoMerging | 根据命中密度合并父块 | 多子块分散命中同一父块 | 中-高 | 树状层级明显 |
| Late Chunking（理论） | 索引时让 chunk 看到完整文档 | embedding 缺少文档全局信息 | 低（但模型受限） | 需长上下文 embedding 模型 |

## 前沿方法：Late Chunking（理论介绍）

### 传统流程的问题
传统 RAG 的流程是先切分，再分别嵌入——每个 chunk 独立通过 embedding 模型，丢失了跨 chunk 的语义关联。

### Late Chunking 思路
Late Chunking 颠倒了这个顺序：
1. 先将**整个文档**送入长上下文 embedding 模型（如 jina-embeddings-v2），获得每个 token 的上下文化表示
2. 再按预设边界切分 token embeddings，对每个 chunk 的 token embeddings 做池化得到 chunk embedding

这样每个 chunk 的向量都见过完整文档上下文，天然缓解了上下文割裂问题。

### 与本章方法的对比

| 维度 | Sentence Window / Small-to-Big / AutoMerging | Late Chunking |
|---|---|---|
| 增强时机 | 检索时（命中后恢复邻域） | 索引时（embedding 阶段） |
| 额外存储 | 需要维护邻居映射 / 父子关系 | 不需要 |
| 模型依赖 | 无特殊要求 | 需要长上下文 embedding 模型 |
| 实现复杂度 | 中 | 低（但模型选择受限） |

### 局限
- 依赖支持长上下文的 embedding 模型（如 jina-embeddings-v2、nomic-embed），智谱 embedding-3 等通用 embedding 模型不直接支持此模式
- 文档超过模型上下文窗口时需要分段处理
- 目前 LangChain 生态无开箱即用的 Late Chunking 组件

### 本节为什么不做代码示例
Late Chunking 需要特殊的 embedding 模型和自定义 tokenizer 操作，与本节统一使用本地 bge-small-zh-v1.5 的可复现实验设定不兼容。此处仅作概念介绍，帮助读者建立还有一类索引时上下文增强的认知。

## 如何选择

- 如果问题经常只差前后两句：先用 Sentence Window。
- 如果文档天然有章节层级：优先 Small-to-Big。
- 如果证据常分散在同一父块多个子块：优先 AutoMerging。
- 若问题本质是多步推理或多轮交互，应转到流程增强或系统增强。
- 如果问题出在 chunk 本身缺少文档语境（脱离上下文后语义不完整），这属于**索引阶段**优化，应回到第 3 章的 CCH / Contextual Retrieval。本章的上下文增强解决的是检索后恢复邻域，而非索引时缺少语境。

## 下一步

如果你发现问题的根源不是上下文不全，而是一次检索流程本身不够——比如需要多步推理、需要先评估检索质量再决定下一步——请继续学习 `2. 流程增强.ipynb`。

### 学习检查点

- 你能区分检索相关但上下文不足与流程不足吗？
- 你能解释 Sentence Window 与 Small-to-Big 的关键差异吗？
- 你知道 AutoMerging 的阈值会如何影响召回上下文长度吗？